## 0. Imports

In [1]:
import sys, os, gc, time, random, json
import torch.nn.functional as F
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, Dataset, TensorDataset, Subset

MONOLITH_ROOT = Path.home() / "project" / "monolith"
sys.path.insert(0, str(MONOLITH_ROOT))

from src.config import get_texas_config, get_adult_config
from src.data.texas_dataset import TexasHospitalExperiment
from src.data.adult_dataset import AdultExperiment
from src.models.tabular_model import build_tabular_model
from src.trainers.dp_dro_trainer import DPDROTrainer
from src.audit.auditor import SteinkeAuditor
from src.audit.subgroup_analyzer import SubgroupAnalyzer
from src.checkpoint.manager import CheckpointManager, RunDecision
from src.utils.metrics import convert_to_serializable

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 2024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("✅ Environment ready.")

Device: cuda
GPU: NVIDIA L40S
✅ Environment ready.


In [2]:
DATASETS_TO_RUN = ['adult']

configs = {
    'texas': get_texas_config(dp_enabled=True, dro_enabled=True),
    'adult': get_adult_config(dp_enabled=True, dro_enabled=True)
}

# Adjust DRO params for tabular data (BCE loss is much smaller than L1 age regression)
for ds_name in DATASETS_TO_RUN:
    configs[ds_name].dro.alpha = 0.005   # Force DRO activation for low-variance BCE loss
    configs[ds_name].dro.eta = 0.1       # Slightly increase step size for faster lambda convergence
    configs[ds_name].dro.lambda_init = 0.001
    configs[ds_name].experiment_name = f"{ds_name}_main_run"
    configs[ds_name].data_dir = MONOLITH_ROOT / "data"
    
print("✅ Configurations initialized for Texas and Adult.")

✅ Configurations initialized for Texas and Adult.


## 1. 

In [3]:
from torch.utils.data import Dataset
import torch

class TabularCanaryDataset:
    """
    Steinke Canary Insertion for Tabular Data.
    """
    def __init__(self, pool_features, pool_subgroups, k, membership, rng):
        self.k = k
        self.membership = membership
        
        # Select k random indices from the pool
        indices = rng.choice(len(pool_features), size=k, replace=False)
        self.canary_features = torch.tensor(pool_features[indices], dtype=torch.float32)
        
        # Assign random binary labels for canaries
        self.canary_labels = torch.tensor(rng.integers(0, 2, size=k), dtype=torch.float32)
        
        self.canary_subgroups = {
            name: torch.tensor(vals[indices], dtype=torch.long) 
            for name, vals in pool_subgroups.items()
        }
        
    def get_in_dataset(self):
        membership_list = self.membership.tolist() if hasattr(self.membership, 'tolist') else list(self.membership)
        in_idxs = [i for i, m in enumerate(membership_list) if m]
        in_features = self.canary_features[in_idxs]
        in_labels = self.canary_labels[in_idxs]
        in_subgroups = {name: vals[in_idxs] for name, vals in self.canary_subgroups.items()}
        return _InTabularCanarySubset(in_features, in_labels, in_subgroups), in_idxs
        
    def get_all_canary_batch(self):
        return self.canary_features, self.canary_labels, self.canary_subgroups
        
    def get_subgroup_labels(self):
        return {name: vals.numpy() for name, vals in self.canary_subgroups.items()}

class _InTabularCanarySubset(Dataset):
    def __init__(self, features, labels, subgroups):
        self.features = features
        self.labels = labels
        self.subgroups = subgroups
        
        n = len(labels)
        self._default_gender = torch.full((n,), -1, dtype=torch.long)
        self._default_race = torch.full((n,), -1, dtype=torch.long)
        
    def __len__(self): 
        return len(self.labels)
        
    def __getitem__(self, i):
        # FIX: Ensure we always return a scalar tensor for subgroups
        gender_tensor = self.subgroups.get('gender', self._default_gender)
        race_tensor = self.subgroups.get('race', self._default_race)
        
        return (
            self.features[i], 
            self.labels[i], 
            gender_tensor[i],  # Now strictly a scalar
            race_tensor[i]     # Now strictly a scalar
        )

print("✅ TabularCanaryDataset defined successfully. You can now run the next cell.")

✅ TabularCanaryDataset defined successfully. You can now run the next cell.


## 2.

In [4]:
# ══════════════════════════════════════════════════════════════════
# 6. FULL TRAINING LOOP WRAPPER (CORRECTED FOR CLASSIFICATION)
# ══════════════════════════════════════════════════════════════════

def run_training_phase(config, phase_name, is_dro, train_ds_full, val_loader, 
                       canary_ds, membership, input_dim, resume_from_epoch=0):
    k = len(membership)
    
    # Build fresh model for this phase
    model = build_tabular_model(input_dim=input_dim, model_type='logreg')
    
    # Save initial parameters for WB scoring
    initial_params = {name: p.detach().clone() for name, p in model.named_parameters()}
    
    # Initialize trainer
    trainer = DPDROTrainer(
        config=config, device=device, model=model,
        full_train_dataset=train_ds_full,
        train_indices=list(range(len(train_ds_full))),
        train_transform=None
    )
    
    # Prepare model with Opacus
    train_loader = DataLoader(
        augmented_train_ds, batch_size=config.train.batch_size,
        shuffle=True, num_workers=0, pin_memory=True
    )
    train_loader = trainer.prepare(train_loader)
    
    # Load checkpoint if resuming
    start_epoch = 1
    # FIX: Use 'error_rate' instead of 'overall_mae'
    history = {
        'train_loss': [], 'error_rate': [], 'gender_gap': [], 
        'race_gap': [], 'epsilon': [], 'lambda_t': []
    }
    
    if resume_from_epoch > 0:
        ckpt = CKPT.load_training_checkpoint(phase_name, epoch=resume_from_epoch)
        if ckpt is not None:
            print(f"  📂 Loading {phase_name} checkpoint from epoch {resume_from_epoch}")
            trainer.get_base_model().load_state_dict(ckpt['model_state_dict'])
            trainer.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch'] + 1
            history = ckpt['history']
            trainer.set_dro_state(ckpt.get('dro_state', {}))
            if 'initial_params' in ckpt.get('extra', {}):
                initial_params = ckpt['extra']['initial_params']
            print(f"  ✅ Resumed. Starting from epoch {start_epoch}.")
    
    # Compute initial canary losses (for Black-Box scoring)
    if start_epoch == 1:
        with torch.no_grad():
            c_feats, c_labs, _ = canary_ds.get_all_canary_batch()
            c_feats, c_labs = c_feats.to(device), c_labs.to(device)
            initial_losses = torch.nn.functional.binary_cross_entropy_with_logits(
                trainer.get_base_model()(c_feats), c_labs, reduction='none'
            ).cpu().numpy()
    else:
        ckpt = CKPT.load_training_checkpoint(phase_name, epoch=resume_from_epoch)
        initial_losses = ckpt.get('extra', {}).get('initial_losses')
        if initial_losses is None:
            with torch.no_grad():
                c_feats, c_labs, _ = canary_ds.get_all_canary_batch()
                initial_losses = torch.nn.functional.binary_cross_entropy_with_logits(
                    trainer.get_base_model()(c_feats.to(device)), c_labs.to(device), reduction='none'
                ).cpu().numpy()
    
    print(f'\n[{phase_name.upper()}] Training epochs {start_epoch} to {config.train.epochs}...')
    start_time = time.time()
    
    try:
        for epoch in range(start_epoch, config.train.epochs + 1):
            # ── DRO Sub-epoch Sampling ──
            if is_dro and epoch > config.dro.warmup_epochs:
                dro_indices = trainer.get_dro_epoch_indices(epoch)
                if dro_indices is not None:
                    # FIX: dro_indices are already relative to train_ds_full (0 to n_train-1).
                    # Do NOT map them through train_idx again.
                    sub_train_ds = Subset(train_ds_full, dro_indices)
                    sub_augmented_ds = ConcatDataset([sub_train_ds, in_canary_ds])
                    
                    train_loader = DataLoader(
                        sub_augmented_ds, batch_size=config.train.batch_size,
                        shuffle=True, num_workers=0, pin_memory=True
                    )
            else:
                train_loader = DataLoader(
                    augmented_train_ds, batch_size=config.train.batch_size,
                    shuffle=True, num_workers=0, pin_memory=True
                )
            
            # Train one epoch
            epoch_loss = trainer.train_epoch(train_loader)
            
            # DRO update (end of epoch)
            if is_dro and epoch > config.dro.warmup_epochs:
                eval_loader = DataLoader(train_ds_full, batch_size=256, shuffle=False, num_workers=0)
                trainer.update_dro(eval_loader)
            
            # Evaluate fairness
            fair_metrics = evaluate_fairness(trainer.get_base_model(), val_loader)
            
            # FIX: Append using classification keys
            history['error_rate'].append(fair_metrics['error_rate'])
            history['train_loss'].append(epoch_loss)
            history['gender_gap'].append(fair_metrics['gender_gap'])
            history['race_gap'].append(fair_metrics['race_gap'])
            history['epsilon'].append(trainer.get_epsilon())
            history['lambda_t'].append(trainer.lambda_t if is_dro else 0.0)
            
            if epoch % 10 == 0 or epoch == start_epoch:
                elapsed = time.time() - start_time
                print(f'  [{phase_name.upper()}] Ep {epoch:2d} | L={epoch_loss:.3f} | '
                      f'Err={fair_metrics["error_rate"]:.3f} | ε={history["epsilon"][-1]:.2f} | '
                      f'λ={history["lambda_t"][-1]:.4f} | {elapsed:.0f}s')
            
            # Save checkpoint every 10 epochs
            if epoch % 10 == 0 or epoch == config.train.epochs:
                CKPT.save_training_checkpoint(
                    phase=phase_name, epoch=epoch, model=trainer.model,
                    optimizer=trainer.optimizer, history=history,
                    dro_state=trainer.get_dro_state(),
                    extra={'initial_losses': initial_losses}
                )
    
    except KeyboardInterrupt:
        current_epoch = epoch if 'epoch' in dir() else start_epoch
        print(f"\n  ⚠️  INTERRUPTED at epoch {current_epoch}! Saving emergency checkpoint...")
        CKPT.save_training_checkpoint(
            phase=phase_name, epoch=current_epoch, model=trainer.model,
            optimizer=trainer.optimizer, history=history,
            dro_state=trainer.get_dro_state(),
            extra={'initial_losses': initial_losses}
        )
        return None
    
    # Compute final canary losses (for Black-Box scoring)
    with torch.no_grad():
        c_feats, c_labs, _ = canary_ds.get_all_canary_batch()
        final_losses = torch.nn.functional.binary_cross_entropy_with_logits(
            trainer.get_base_model()(c_feats.to(device)), c_labs.to(device), reduction='none'
        ).cpu().numpy()
    
    # Compute White-Box scores
    print(f"  Computing White-Box scores...")
    wb_model = build_tabular_model(input_dim=input_dim, model_type='logreg')
    wb_model.load_state_dict(trainer.get_base_model().state_dict())
    wb_model.to(device)
    wb_model.eval()
    
    final_params = {name: p.detach() for name, p in wb_model.named_parameters()}
    delta_w = torch.cat([(final_params[name] - initial_params[name].to(device)).view(-1) for name in final_params])
    
    c_feats, c_labs, _ = canary_ds.get_all_canary_batch()
    c_feats, c_labs = c_feats.to(device), c_labs.to(device)
    wb_scores = np.zeros(k, dtype=np.float64)
    
    for s in range(0, k, 64):
        e = min(s + 64, k)
        sub_feats = c_feats[s:e].clone().detach().requires_grad_(True)
        sub_labs = c_labs[s:e]
        
        wb_model.zero_grad()
        preds = wb_model(sub_feats)
        losses = torch.nn.functional.binary_cross_entropy_with_logits(preds, sub_labs, reduction='sum')
        
        grads = torch.autograd.grad(losses, list(wb_model.parameters()))
        grad_flat = torch.cat([g.view(-1) for g in grads])
        
        clip_f = min(1.0, config.dp.max_grad_norm / (grad_flat.norm().item() + 1e-8))
        wb_scores[s:e] = torch.dot(delta_w, grad_flat * clip_f).cpu().numpy()
    
    print(f'  [{phase_name.upper()}] Done in {time.time()-start_time:.0f}s')
    
    return {
        'model': trainer.get_base_model(),
        'bb_scores': initial_losses - final_losses,
        'wb_scores': wb_scores,
        'history': history,
        'epsilon_spent': history['epsilon'][-1],
        'final_metrics': fair_metrics
    }


def evaluate_fairness(model, loader):
    """Evaluate model fairness on validation set (Classification)."""
    model.eval()
    gender_loss, gender_count = {0:0.0, 1:0.0}, {0:0, 1:0}
    race_loss, race_count = {}, {}
    total_loss, total_count = 0.0, 0
    
    with torch.no_grad():
        for feats, labs, genders, races in loader:
            feats, labs = feats.to(device), labs.to(device)
            preds = model(feats)
            # Error rate (1 - accuracy) as proxy for MAE in classification
            err = (torch.sigmoid(preds).round() != labs).float().cpu()
            valid = (genders >= 0)
            total_loss += err[valid].sum().item()
            total_count += valid.sum().item()
            
            for g in (0, 1):
                mask = (genders == g) & valid
                if mask.any():
                    gender_loss[g] += err[mask].sum().item()
                    gender_count[g] += mask.sum().item()
                    
            for r in torch.unique(races[valid]):
                r_val = r.item()
                if r_val not in race_loss: race_loss[r_val] = 0.0; race_count[r_val] = 0
                mask = (races == r_val) & valid
                race_loss[r_val] += err[mask].sum().item()
                race_count[r_val] += mask.sum().item()
                
    male_err = gender_loss[0] / max(gender_count[0], 1)
    female_err = gender_loss[1] / max(gender_count[1], 1)
    gender_gap = abs(male_err - female_err)
    
    race_errs = [race_loss[r] / max(race_count[r], 1) for r in race_loss if race_count[r] > 10]
    race_gap = max(race_errs) - min(race_errs) if len(race_errs) > 1 else 0.0
    
    return {
        'error_rate': total_loss / max(total_count, 1),
        'male_error': male_err,
        'female_error': female_err,
        'gender_gap': gender_gap,
        'race_gap': race_gap,
    }

print("✅ Training loop wrapper defined (corrected for classification).")

✅ Training loop wrapper defined (corrected for classification).


## 3.

In [5]:
all_final_results = {}

for ds_name in DATASETS_TO_RUN:
    print('\n' + '█' * 70)
    print(f'  DATASET: {ds_name.upper()}')
    print('█' * 70)
    
    CFG = configs[ds_name]
    output_dir = CFG.get_output_dir()
    CKPT = CheckpointManager(output_dir, experiment_name=CFG.experiment_name)
    
    # Load Data
    if ds_name == 'texas':
        experiment_ds = TexasHospitalExperiment(CFG.data_dir)
    else:
        experiment_ds = AdultExperiment(CFG.data_dir)
        
    full_ds = experiment_ds.load()
    N = len(full_ds)
    print(f"  Loaded {N} samples.")
    
    k = CFG.audit.k
    audit_rng = np.random.default_rng(CFG.audit.seed)
    all_indices = list(range(N))
    audit_rng.shuffle(all_indices)
    
    canary_pool_idx = all_indices[:k*2]
    remaining_idx = all_indices[k*2:]
    
    n_val = int(len(remaining_idx) * CFG.train.val_frac)
    gen = torch.Generator().manual_seed(CFG.train.seed)
    train_idx, val_idx = torch.utils.data.random_split(remaining_idx, [len(remaining_idx)-n_val, n_val], generator=gen)
    train_idx, val_idx = list(train_idx), list(val_idx)
    
    membership = torch.tensor(audit_rng.integers(0, 2, size=k).astype(bool), dtype=torch.bool)
    
    # Create Datasets
    train_ds_full = Subset(full_ds, train_idx)
    val_ds = Subset(full_ds, val_idx)
    
    # Canary Pool
    pool_features = experiment_ds._features[canary_pool_idx]
    pool_subgroups = {name: vals[canary_pool_idx] for name, vals in experiment_ds._subgroups.items()}
    
    canary_ds = TabularCanaryDataset(pool_features, pool_subgroups, k, membership, audit_rng)
    in_canary_ds, _ = canary_ds.get_in_dataset()
    augmented_train_ds = ConcatDataset([train_ds_full, in_canary_ds])
    
    val_loader = DataLoader(val_ds, batch_size=CFG.train.batch_size, shuffle=False, num_workers=0)
    input_dim = experiment_ds.input_dim
    
    # ERM
    print('\n' + '═' * 60 + '\n  PHASE 1: DP-SGD + ERM\n' + '═' * 60)
    cfg_erm = get_texas_config() if ds_name == 'texas' else get_adult_config()
    cfg_erm.dp = CFG.dp; cfg_erm.dro.enabled = False; cfg_erm.train = CFG.train
    
    res_erm = run_training_phase(cfg_erm, 'erm', False, train_ds_full, val_loader, canary_ds, membership, input_dim)
    
    # DRO
    print('\n' + '═' * 60 + '\n  PHASE 2: DP-SGD + DRO\n' + '═' * 60)
    cfg_dro = get_texas_config() if ds_name == 'texas' else get_adult_config()
    cfg_dro.dp = CFG.dp; cfg_dro.dro = CFG.dro; cfg_dro.train = CFG.train
    
    res_dro = run_training_phase(cfg_dro, 'dro', True, train_ds_full, val_loader, canary_ds, membership, input_dim)
    
    # Audit
    print('\n' + '═' * 60 + '\n  PHASE 3: Privacy Audit\n' + '═' * 60)
    auditor = SteinkeAuditor(gamma=CFG.audit.gamma)
    bb_erm = auditor.audit(res_erm['bb_scores'], membership.numpy())
    wb_erm = auditor.audit(res_erm['wb_scores'], membership.numpy())
    bb_dro = auditor.audit(res_dro['bb_scores'], membership.numpy())
    wb_dro = auditor.audit(res_dro['wb_scores'], membership.numpy())
    
    print(f"  ERM  -> BB AUC: {bb_erm['auc']:.3f}, WB AUC: {wb_erm['auc']:.3f}")
    print(f"  DRO  -> BB AUC: {bb_dro['auc']:.3f}, WB AUC: {wb_dro['auc']:.3f}")
    
    # Subgroup
    canary_subgroup_labels = canary_ds.get_subgroup_labels()
    analyzer = SubgroupAnalyzer({'race': ['R0', 'R1', 'R2', 'R3', 'R4'], 'gender': ['Male', 'Female']})
    
    erm_sub = analyzer.analyze(res_erm['wb_scores'], membership.numpy(), canary_subgroup_labels)
    dro_sub = analyzer.analyze(res_dro['wb_scores'], membership.numpy(), canary_subgroup_labels)
    
    all_final_results[ds_name] = {
        'erm': {'bb': bb_erm, 'wb': wb_erm, 'metrics': res_erm['final_metrics'], 'sub': erm_sub},
        'dro': {'bb': bb_dro, 'wb': wb_dro, 'metrics': res_dro['final_metrics'], 'sub': dro_sub}
    }

print("\n✅ All datasets processed!")


██████████████████████████████████████████████████████████████████████
  DATASET: ADULT
██████████████████████████████████████████████████████████████████████
  Loaded 10000 samples.

════════════════════════════════════════════════════════════
  PHASE 1: DP-SGD + ERM
════════════════════════════════════════════════════════════


/home/govindu.srimaan/mlenv/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(



[ERM] Training epochs 1 to 50...
  [ERM] Ep  1 | L=0.742 | Err=0.499 | ε=0.97 | λ=0.0000 | 1s
  [ERM] Ep 10 | L=0.706 | Err=0.474 | ε=2.38 | λ=0.0000 | 5s
  [ERM] Ep 20 | L=0.701 | Err=0.487 | ε=3.35 | λ=0.0000 | 11s
  [ERM] Ep 30 | L=0.700 | Err=0.499 | ε=4.13 | λ=0.0000 | 15s
  [ERM] Ep 40 | L=0.700 | Err=0.495 | ε=4.81 | λ=0.0000 | 21s
  [ERM] Ep 50 | L=0.700 | Err=0.490 | ε=5.44 | λ=0.0000 | 27s
  Computing White-Box scores...
  [ERM] Done in 27s

════════════════════════════════════════════════════════════
  PHASE 2: DP-SGD + DRO
════════════════════════════════════════════════════════════

[DRO] Training epochs 1 to 50...


/home/govindu.srimaan/mlenv/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


  [DRO] Ep  1 | L=0.724 | Err=0.506 | ε=0.97 | λ=0.0010 | 0s
  [DRO] Ep 10 | L=0.703 | Err=0.488 | ε=2.12 | λ=0.0096 | 4s
  [DRO] Ep 20 | L=0.700 | Err=0.490 | ε=2.73 | λ=0.0193 | 7s
  [DRO] Ep 30 | L=0.697 | Err=0.490 | ε=3.25 | λ=0.0229 | 11s
  [DRO] Ep 40 | L=0.695 | Err=0.501 | ε=3.71 | λ=0.0233 | 15s
  [DRO] Ep 50 | L=0.694 | Err=0.509 | ε=4.13 | λ=0.0230 | 19s
  Computing White-Box scores...
  [DRO] Done in 19s

════════════════════════════════════════════════════════════
  PHASE 3: Privacy Audit
════════════════════════════════════════════════════════════
  ERM  -> BB AUC: 0.455, WB AUC: 0.502
  DRO  -> BB AUC: 0.503, WB AUC: 0.517

✅ All datasets processed!


In [6]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4: Disparate Vulnerability Analysis (Tabular)
# ══════════════════════════════════════════════════════════════════
print('\n' + '═' * 60)
print('  PHASE 4: Disparate Vulnerability Analysis')
print('═' * 60)

# Get subgroup labels for the canaries
canary_subgroup_labels = canary_ds.get_subgroup_labels()

# Dynamically determine subgroup names based on the synthetic/real data
subgroup_names = {}
for sg_name, labels in canary_subgroup_labels.items():
    unique_vals = np.unique(labels)
    subgroup_names[sg_name] = [f"{sg_name}_{v}" for v in unique_vals]

analyzer = SubgroupAnalyzer(subgroup_names)

# Analyze ERM subgroup vulnerability
print("\n[ERM Subgroup AUC]")
erm_subgroup_results = analyzer.analyze(
    res_erm['wb_scores'], membership.numpy(), canary_subgroup_labels
)
erm_disparities = analyzer.compute_disparity(erm_subgroup_results)

for sg_name, groups in erm_subgroup_results.items():
    print(f"  {sg_name}:")
    for g_name, stats in groups.items():
        print(f"    {g_name:10s}: AUC={stats['auc']:.3f} (n_in={stats['n_in']}, n_out={stats['n_out']})")

# Analyze DRO subgroup vulnerability
print("\n[DRO Subgroup AUC]")
dro_subgroup_results = analyzer.analyze(
    res_dro['wb_scores'], membership.numpy(), canary_subgroup_labels
)
dro_disparities = analyzer.compute_disparity(dro_subgroup_results)

for sg_name, groups in dro_subgroup_results.items():
    print(f"  {sg_name}:")
    for g_name, stats in groups.items():
        print(f"    {g_name:10s}: AUC={stats['auc']:.3f} (n_in={stats['n_in']}, n_out={stats['n_out']})")

print("\n✅ Subgroup analysis complete! Now re-run the 'Save Final Results' cell below.")


════════════════════════════════════════════════════════════
  PHASE 4: Disparate Vulnerability Analysis
════════════════════════════════════════════════════════════

[ERM Subgroup AUC]
  race:
    race_0    : AUC=0.470 (n_in=21, n_out=33)
    race_1    : AUC=0.448 (n_in=12, n_out=16)
    race_2    : AUC=0.604 (n_in=28, n_out=20)
    race_3    : AUC=0.519 (n_in=18, n_out=15)
    race_4    : AUC=0.386 (n_in=15, n_out=22)
  gender:
    gender_0  : AUC=0.525 (n_in=39, n_out=47)
    gender_1  : AUC=0.479 (n_in=55, n_out=59)

[DRO Subgroup AUC]
  race:
    race_0    : AUC=0.543 (n_in=21, n_out=33)
    race_1    : AUC=0.365 (n_in=12, n_out=16)
    race_2    : AUC=0.560 (n_in=28, n_out=20)
    race_3    : AUC=0.615 (n_in=18, n_out=15)
    race_4    : AUC=0.450 (n_in=15, n_out=22)
  gender:
    gender_0  : AUC=0.548 (n_in=39, n_out=47)
    gender_1  : AUC=0.494 (n_in=55, n_out=59)

✅ Subgroup analysis complete! Now re-run the 'Save Final Results' cell below.


## 4.

In [7]:
# ══════════════════════════════════════════════════════════════════
# 10. SAVE FINAL RESULTS & SUMMARY (CORRECTED FOR CLASSIFICATION)
# ══════════════════════════════════════════════════════════════════

print('\n' + '═' * 60)
print('  SAVING FINAL RESULTS')
print('═' * 60)

final_results = {
    'timestamp': datetime.now().isoformat(),
    'experiment_name': CFG.experiment_name,
    'config': convert_to_serializable(CFG.to_dict()),
    'erm': {
        'history': convert_to_serializable(res_erm['history']),
        'epsilon_spent': float(res_erm['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_erm),
        'wb_audit': convert_to_serializable(wb_erm),
        'final_metrics': convert_to_serializable(res_erm['final_metrics']),
        'subgroup_auc': convert_to_serializable(erm_subgroup_results),
        'disparities': convert_to_serializable(erm_disparities),
    },
    'dro': {
        'history': convert_to_serializable(res_dro['history']),
        'epsilon_spent': float(res_dro['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_dro),
        'wb_audit': convert_to_serializable(wb_dro),
        'final_metrics': convert_to_serializable(res_dro['final_metrics']),
        'final_lambda_t': float(res_dro['history']['lambda_t'][-1]),
        'subgroup_auc': convert_to_serializable(dro_subgroup_results),
        'disparities': convert_to_serializable(dro_disparities),
    },
    'comparison': {
        'error_rate_increase': float(res_dro['final_metrics']['error_rate'] - res_erm['final_metrics']['error_rate']),
        'gender_gap_reduction_pct': float(
            (res_erm['final_metrics']['gender_gap'] - res_dro['final_metrics']['gender_gap'])
            / max(res_erm['final_metrics']['gender_gap'], 0.001) * 100
        ),
        'race_gap_reduction_pct': float(
            (res_erm['final_metrics']['race_gap'] - res_dro['final_metrics']['race_gap'])
            / max(res_erm['final_metrics']['race_gap'], 0.001) * 100
        ),
        'wb_auc_delta': float(dro_disparities.get('race', {}).get('gap', 0) - erm_disparities.get('race', {}).get('gap', 0)),
    }
}

CKPT.save_final_results(final_results)

# Mark experiment as complete
metadata = CKPT.load_metadata() or {}
metadata['status'] = 'complete'
metadata['completed_phases'] = list(set(metadata.get('completed_phases', []) + ['complete']))
metadata['end_time'] = datetime.now().isoformat()
CKPT.save_metadata(metadata)

print(f"✅ Final results saved to: {output_dir / 'checkpoints' / 'final_results.json'}")

# Print summary table
print('\n' + '═' * 60)
print(f'  EXPERIMENT SUMMARY: {ds_name.upper()}')
print('═' * 60)
print(f"{'Metric':<25} {'ERM':<15} {'DRO':<15} {'Δ (DRO-ERM)':<15}")
print('─' * 60)
print(f"{'Error Rate':<25} {res_erm['final_metrics']['error_rate']:<15.3f} {res_dro['final_metrics']['error_rate']:<15.3f} {res_dro['final_metrics']['error_rate'] - res_erm['final_metrics']['error_rate']:<+15.3f}")
print(f"{'Gender Gap':<25} {res_erm['final_metrics']['gender_gap']:<15.3f} {res_dro['final_metrics']['gender_gap']:<15.3f} {res_dro['final_metrics']['gender_gap'] - res_erm['final_metrics']['gender_gap']:<+15.3f}")
print(f"{'Race Gap':<25} {res_erm['final_metrics']['race_gap']:<15.3f} {res_dro['final_metrics']['race_gap']:<15.3f} {res_dro['final_metrics']['race_gap'] - res_erm['final_metrics']['race_gap']:<+15.3f}")
print(f"{'BB AUC':<25} {bb_erm['auc']:<15.3f} {bb_dro['auc']:<15.3f} {bb_dro['auc'] - bb_erm['auc']:<+15.3f}")
print(f"{'WB AUC':<25} {wb_erm['auc']:<15.3f} {wb_dro['auc']:<15.3f} {wb_dro['auc'] - wb_erm['auc']:<+15.3f}")
print(f"{'BB ε_lb':<25} {bb_erm['eps_lb']:<15.3f} {bb_dro['eps_lb']:<15.3f} {bb_dro['eps_lb'] - bb_erm['eps_lb']:<+15.3f}")
print(f"{'WB ε_lb':<25} {wb_erm['eps_lb']:<15.3f} {wb_dro['eps_lb']:<15.3f} {wb_dro['eps_lb'] - wb_erm['eps_lb']:<+15.3f}")
print(f"{'Final λ_t':<25} {'N/A':<15} {res_dro['history']['lambda_t'][-1]:<15.6f} {'':<15}")
print('═' * 60)

print(f"\n🎉 {ds_name.upper()} Experiment complete! Check the JSON file for full details.")


════════════════════════════════════════════════════════════
  SAVING FINAL RESULTS
════════════════════════════════════════════════════════════
✅ Final results saved to: experiments/adult_main_run/adult_dro_dp_sigma1.0/checkpoints/final_results.json

════════════════════════════════════════════════════════════
  EXPERIMENT SUMMARY: ADULT
════════════════════════════════════════════════════════════
Metric                    ERM             DRO             Δ (DRO-ERM)    
────────────────────────────────────────────────────────────
Error Rate                0.490           0.509           +0.019         
Gender Gap                0.019           0.015           -0.004         
Race Gap                  0.116           0.134           +0.018         
BB AUC                    0.455           0.503           +0.048         
WB AUC                    0.502           0.517           +0.015         
BB ε_lb                   0.000           0.000           +0.000         
WB ε_lb           